# Качество кристалла по ширине и положению фононов

Для каждой точки: центр и ширина (FWHM) TO и LO, собственная ширина без
аппаратного уширения прибора (~7 см⁻¹) и доля LO = S(LO) / (S(TO) + S(LO)).
На (100) запрещён TO; чем больше беспорядка, тем сильнее он «просыпается»,
доля LO падает, LO уширяется и сдвигается вниз.

Нажмите **Run All**. Файлы `../data/raw/own/*.txt` появляются после `scripts/import_own.py`.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from ramancore.io import read_spectrum
from ramancore.semiconductors import phonon_quality

INSTRUMENT_FWHM = 7.0  # аппаратная ширина Confotec Duo (6.8–7.5 см⁻¹ по статье)

## 1. Эталон: нелегированный i-GaAs (100) при 294 K

In [ ]:
x, y, _ = read_spectrum("../data/raw/own/own_gaas_vs182_294K.txt")
ref = phonon_quality(x, y, "GaAs", instrument_fwhm=INSTRUMENT_FWHM)
reference_lo = ref["table"].loc["LO", "center"]
print(f"LO эталона = {reference_lo:.2f} см⁻¹, доля LO = {ref['lo_fraction']:.2f}")
ref["table"].round(2)

## 2. Серия «дефектность» (точки а–ж, GaAs:Te)

In [ ]:
points = {  # файл → что в этой точке (по рисунку статьи)
    "own_gaas_te_A": "а поликристалл",
    "own_gaas_te_B": "б центр ламели",
    "own_gaas_te_V": "в дислокация",
    "own_gaas_te_G": "г скопление дислокаций",
    "own_gaas_te_D": "д малоугловая граница",
    "own_gaas_te_E": "е край малоугловой границы",
    "own_gaas_ar97": "ж монокристалл",
}
rows, fits = [], {}
for name, label in points.items():
    x, y, _ = read_spectrum(f"../data/raw/own/{name}.txt")
    res = phonon_quality(x, y, "GaAs", instrument_fwhm=INSTRUMENT_FWHM, reference_lo=reference_lo)
    t = res["table"]
    rows.append({"точка": label,
                 "TO, см⁻¹": t.loc["TO", "center"], "FWHM TO": t.loc["TO", "fwhm"],
                 "LO, см⁻¹": t.loc["LO", "center"], "FWHM LO": t.loc["LO", "fwhm"],
                 "собств. FWHM LO": t.loc["LO", "intrinsic_fwhm"],
                 "сдвиг LO": res["lo_shift"],
                 "доля LO": res["lo_fraction"], "±": res["lo_fraction_err"]})
    fits[label] = (x, y, res["fit"])
pd.DataFrame(rows).round(2)

In [ ]:
fig = go.Figure()
for label, (x, y, fit) in fits.items():
    keep = (x > 200) & (x < 360)
    fig.add_scatter(x=x[keep], y=y[keep] / y[keep].max(), name=label)
fig.update_layout(xaxis_title="Рамановский сдвиг, см⁻¹", yaxis_title="Нормированная интенсивность")
fig.show()

## Как читать

* Доля LO близко к 1 — совершенный (100); около 0.2 и ниже — сильное разупорядочение или поликристалл.
* Сдвиг LO вниз на несколько см⁻¹ и уширение LO — дефекты и напряжения.
* В легированных образцах (ж, Ар-97) в TO входит мода L−, поэтому ширина TO там завышена.
* Собственная ширина при FWHM близкой к 7 см⁻¹ определяется грубо.

In [ ]:
for w in ref["warnings"]:
    print("•", w)